In [1]:
import requests
import xml.etree.ElementTree as ET
import time
import json
import xmltodict
import re
from urllib.parse import urlparse

In [2]:
# get record ids
index_url = "https://www.re3data.org/api/v1/repositories"
response = requests.get(index_url)
root = ET.fromstring(response.content)

all_repo_ids = [repo.find('id').text for repo in root.findall('repository')]

print(f"Total re3data repositories to check: {len(all_repo_ids)}")

Total re3data repositories to check: 3505


In [3]:
# get full records of ids
re3_full_records = []
skipped_count = 0

# load mapping from the other file
with open("../fs_to_re3_mapping.json", "r", encoding="utf-8") as f:
    fs_to_re3_mapping = json.load(f)

known_ids = set(fs_to_re3_mapping.values())

for i, r3d_id in enumerate(all_repo_ids):
    # DEDUPLICATION CHECK
    if r3d_id in known_ids:
        skipped_count += 1
        continue

    detail_url = f"https://www.re3data.org/api/v1/repository/{r3d_id}"

    try:
        resp = requests.get(detail_url)
        if resp.status_code == 200:
            re3_full_records.append(resp.text)

        # pause after every 10 records
        if i % 10 == 0:
            print(f"Progress: {i}/{len(all_repo_ids)} downloaded...")
            time.sleep(0.5)

    except Exception as e:
        print(f"Error downloading {r3d_id}: {e}")

print(f"Finished! Downloaded {len(re3_full_records)} new records. Skipped {skipped_count} duplicates.")


Progress: 0/3505 downloaded...
Progress: 10/3505 downloaded...
Progress: 20/3505 downloaded...
Progress: 30/3505 downloaded...
Progress: 40/3505 downloaded...
Progress: 50/3505 downloaded...
Progress: 60/3505 downloaded...
Progress: 70/3505 downloaded...
Progress: 80/3505 downloaded...
Progress: 90/3505 downloaded...
Progress: 100/3505 downloaded...
Progress: 110/3505 downloaded...
Progress: 120/3505 downloaded...
Progress: 130/3505 downloaded...
Progress: 140/3505 downloaded...
Progress: 150/3505 downloaded...
Progress: 160/3505 downloaded...
Progress: 180/3505 downloaded...
Progress: 190/3505 downloaded...
Progress: 200/3505 downloaded...
Progress: 210/3505 downloaded...
Progress: 220/3505 downloaded...
Progress: 240/3505 downloaded...
Progress: 250/3505 downloaded...
Progress: 260/3505 downloaded...
Progress: 270/3505 downloaded...
Progress: 280/3505 downloaded...
Progress: 290/3505 downloaded...
Progress: 300/3505 downloaded...
Progress: 310/3505 downloaded...
Progress: 320/3505 do

In [5]:
with open("data/re3data_all_records.json", "w", encoding="utf-8") as f:
    json.dump(re3_full_records, f)

with open("data/re3data_all_records.json", "r") as f:
    re3_all_records = json.load(f)

In [6]:
# helper function to print xml
def print_structure(element, level=0):
    indent = "  " * level
    tag = element.tag.split('}')[-1]

    text = element.text.strip() if element.text else ""
    if text:
        print(f"{indent}<{tag}>: {text[:70]}{'...' if len(text) > 70 else ''}")
    else:
        print(f"{indent}<{tag}>")

    for child in element:
        print_structure(child, level + 1)

In [7]:
# filtering for covid using the same exact keywords as for fairsharing
covid_keywords = ["covid", "coronavirus", "coronaviridae", "sars-cov-2", "mers-cov", "hcov-sars", "hcov-19", "2019-ncov"]
re3_covid = []

for xml_text in re3_all_records:
    root = ET.fromstring(xml_text)

    subjects = [s.text for s in root.findall(".//{*}subject") if s.text]
    is_life_science = any(s.startswith('2') for s in subjects)

    if is_life_science:
        name_el = root.find(".//{*}repositoryName")
        desc_el = root.find(".//{*}description")

        name = name_el.text if name_el is not None else ""
        desc = desc_el.text if desc_el is not None else ""

        keywords = [k.text for k in root.findall(".//{*}keyword") if k.text]

        full_text_blob = (name + " " + desc + " " + " ".join(keywords)).lower()

        if any(key in full_text_blob for key in covid_keywords):
            id_el = root.find(".//{*}re3data.orgIdentifier")
            repo_id = id_el.text if id_el is not None else None

            re3_covid.append({
                "id": repo_id,
                "name": name,
                "subjects": subjects,
                "raw_xml": xml_text
            })

print(f"Found {len(re3_covid)} potential COVID repositories in the Life Sciences.")


Found 18 potential COVID repositories in the Life Sciences.


In [8]:
# look at subjects
subject_counts = {}

for record in re3_covid:
    subjects = record.get('subjects', [])
    if subjects:
        sorted_subjects = sorted(subjects, key=lambda s: len(s.split(' ')[0]), reverse=True)
        most_specific = sorted_subjects[0]
        subject_counts[most_specific] = subject_counts.get(most_specific, 0) + 1

print("\n--- Breakdown of repositories by subject ---")
for sub, count in sorted(subject_counts.items(), key=lambda item: item[1], reverse=True):
    print(f"{sub}: {count} repositories")



--- Breakdown of repositories by subject ---
20101 Biochemistry: 4 repositories
20501 Epidemiology, Medical Biometry, Medical Informatics: 3 repositories
20105 General Genetics: 2 repositories
1 Humanities and Social Sciences: 2 repositories
20404 Virology: 2 repositories
20606 Cognitive Neuroscience and Neuroimaging: 1 repositories
204 Microbiology, Virology and Immunology: 1 repositories
20532 Biomedical Technology and Medical Physics: 1 repositories
20103 Cell Biology: 1 repositories
22 Medicine: 1 repositories


In [15]:
re3_covid

[{'id': 'r3d100010283',
  'name': 'Gene Expression Omnibus',
  'subjects': ['2 Life Sciences',
   '201 Basic Biological and Medical Research',
   '20105 General Genetics',
   '21 Biology'],
  'raw_xml': '<?xml version="1.0" encoding="utf-8"?>\n<!--re3data.org Schema for the Description of Research Data Repositories. Version 2.2, December 2014. doi:10.2312/re3.006-->\n<r3d:re3data xmlns:r3d="http://www.re3data.org/schema/2-2" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.re3data.org/schema/2-2 http://schema.re3data.org/2-2/re3dataV2-2.xsd">\n    <r3d:repository>\n        <r3d:re3data.orgIdentifier>r3d100010283</r3d:re3data.orgIdentifier>\n        <r3d:repositoryName language="eng">Gene Expression Omnibus</r3d:repositoryName>\n                    <r3d:additionalName language="eng">GEO</r3d:additionalName>\n                <r3d:repositoryURL>https://www.ncbi.nlm.nih.gov/geo/</r3d:repositoryURL>\n                    <r3d:repositoryIdentifier>10.25504/

In [13]:
sample_xml = re3_covid[0]["raw_xml"]
sample_root = ET.fromstring(sample_xml)
print_structure(sample_root)

<re3data>
  <repository>
    <re3data.orgIdentifier>: r3d100010283
    <repositoryName>: Gene Expression Omnibus
    <additionalName>: GEO
    <repositoryURL>: https://www.ncbi.nlm.nih.gov/geo/
    <repositoryIdentifier>: 10.25504/FAIRsharing.5hc8vt
    <repositoryIdentifier>: OMICS_01030
    <repositoryIdentifier>: SCR_005012
    <repositoryIdentifier>: nif-0000-00142
    <description>: Gene Expression Omnibus: a public functional genomics data repository ...
    <repositoryContact>: geo@ncbi.nlm.nih.gov
    <type>: disciplinary
    <size>: Platforms 24.993 ; Samples 5.736.661 ; Series 84.386; DataSets 4.348
    <startDate>: 2002
    <repositoryLanguage>: eng
    <subject>: 2 Life Sciences
    <subject>: 201 Basic Biological and Medical Research
    <subject>: 20105 General Genetics
    <subject>: 21 Biology
    <missionStatementURL>: https://www.ncbi.nlm.nih.gov/geo/info/overview.html
    <contentType>: Archived data
    <contentType>: Images
    <contentType>: Plain text
    <conten

In [17]:
import pandas as pd
import xml.etree.ElementTree as ET

NS = {"r3d": "http://www.re3data.org/schema/2-2"}

def get_text(root, path):
    el = root.find(path, NS)
    return el.text.strip() if el is not None and el.text else None

def get_texts(root, path):
    vals = []
    for el in root.findall(path, NS):
        if el is not None and el.text:
            vals.append(el.text.strip())
    return vals

def strip_subject_number(subject):
    parts = subject.strip().split(" ", 1)
    if len(parts) == 2 and parts[0].isdigit():
        return parts[1].strip()
    return subject.strip()

def parse_re3_record(xml_text):
    root = ET.fromstring(xml_text)

    subjects_raw = get_texts(root, ".//r3d:subject")
    subjects_clean = [strip_subject_number(s) for s in subjects_raw]

    content_types = get_texts(root, ".//r3d:contentType")
    keywords = get_texts(root, ".//r3d:keyword")

    institutions = root.findall(".//r3d:institution", NS)

    data = {
        "name": get_text(root, ".//r3d:repositoryName"),
        "additional_name": get_text(root, ".//r3d:additionalName"),
        "re3_id": get_text(root, ".//r3d:re3data.orgIdentifier"),
        "url": get_text(root, ".//r3d:repositoryURL"),
        "description": get_text(root, ".//r3d:description"),
        "contact_email": get_text(root, ".//r3d:repositoryContact"),
        "year_created": get_text(root, ".//r3d:startDate"),
        "subjects": "; ".join(subjects_clean),
        "content_types": "; ".join(content_types),
        "keywords": "; ".join(keywords),
        "data_license_name": get_text(root, ".//r3d:dataLicenseName"),
        "data_license_url": get_text(root, ".//r3d:dataLicenseURL"),
    }

    for i, inst in enumerate(institutions, start=1):
        data[f"institution_name{i}"] = get_text(inst, ".//r3d:institutionName")
        data[f"institution_country{i}"] = get_text(inst, ".//r3d:institutionCountry")
        data[f"institution_responsibility_type{i}"] = get_text(inst, ".//r3d:responsibilityType")
        data[f"institution_type{i}"] = get_text(inst, ".//r3d:institutionType")

    return data

records = [parse_re3_record(rec["raw_xml"]) for rec in re3_covid]
df_re3_covid = pd.DataFrame(records)

df_re3_covid


,name,additional_name,re3_id,url,description,contact_email,year_created,subjects,content_types,keywords,...,institution_responsibility_type9,institution_type9,institution_name10,institution_country10,institution_responsibility_type10,institution_type10,institution_name11,institution_country11,institution_responsibility_type11,institution_type11
0,Gene Expression Omnibus,GEO,r3d100010283,https://www.ncbi.nlm.nih.gov/geo/,Gene Expression Omnibus: a public functional g...,geo@ncbi.nlm.nih.gov,2002,Life Sciences; Basic Biological and Medical Re...,Archived data; Images; Plain text; Scientific ...,COVID-19; Gene Expression; Genetic Regulation;...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Canada's Michael Smith Genome Sciences Centre,GSC,r3d100010449,https://www.bcgsc.ca/,We are a leading international centre for geno...,https://www.bcgsc.ca/contact-us,1999-01-01,Life Sciences; Basic Biological and Medical Re...,Images; Scientific and statistical data format...,Atlantic salmon; COVID-19; SARS Coronavirus; b...,...,funding,non-profit,Western Economic Diversification Canada,CAN,funding,non-profit,NaN,NaN,NaN,NaN
2,UniProtKB,UniProtKnowledgebase,r3d100011521,https://www.uniprot.org/uniprotkb/,The UniProt Knowledgebase (UniProtKB) is the c...,https://www.uniprot.org/contact,None,Life Sciences; Basic Biological and Medical Re...,Raw data; Scientific and statistical data form...,COVID-19; biology; cellular component; disease...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,International Neuroimaging Data-sharing Initia...,INDI,r3d100011555,https://fcon_1000.projects.nitrc.org/,INDI was formed as a next generation FCP effor...,Michael.Milham@childmind.org,2009-12-11,Life Sciences; Neurosciences; Cognitive Neuros...,Archived data; Raw data; Scientific and statis...,COVID-19; mental health; neuroimaging,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,LabKey Open Research Portal,formerly: Zika Open-Research Portal,r3d100012059,https://openresearch.labkey.com/project/home/b...,"In response to emerging pathogens, LabKey laun...",https://www.labkey.com/about/contact/,2016,Life Sciences; Basic Biological and Medical Re...,Images; Raw data; Scientific and statistical d...,ZEST; fetal abnormities; microcephaly; nonhuma...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,SARS-CoV-2 Data Hub,NCBI SARS-CoV,r3d100012192,https://www.ncbi.nlm.nih.gov/labs/virus/vssi/#...,This Web resource provides data and informatio...,genomes@ncbi.nlm.nih.gov,None,"Life Sciences; Microbiology, Virology and Immu...",Databases; Raw data; Scientific and statistica...,COVID-19; RNA; SARS; genomics; nucleotides; se...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,VIPERdb,Virus Particle Explorer database,r3d100012362,https://viperdb.org/,VIPERdb is a database for icosahedral virus ca...,https://viperdb.org/ContactUs.php,None,Life Sciences; Medicine; Medicine; Biomedical ...,Images; Raw data; Scientific and statistical d...,COVID-19; Phi-Psi diagrams; computational biol...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Scholars' Mine,None,r3d100012692,https://scholarsmine.mst.edu/,Scholars' Mine is an online collection of scho...,https://scholarsmine.mst.edu/contact.html,None,Humanities and Social Sciences; Life Sciences;...,Audiovisual data; Scientific and statistical d...,COVID-19; multidisciplinary,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Immune Epitope Database,IEDB,r3d100012702,https://www.iedb.org/,IEDB offers easy searching of experimental dat...,https://help.iedb.org/hc/en-us/requests/new,2004,Life Sciences; Basic Biological and Medical Re...,Archived data; Raw data; Software applications...,COVID-19; allergenes; alloantigenes; autoimmun...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Mass Spectrometry Interactive Virtual Environment,MassIVE,r3d100012858,https://massive.ucsd.edu/ProteoSAFe/static/mas...,MassIVE is a community resource developed by t...,ccms-web@cs.ucsd.edu,None,Life Sciences; Basic Biological and Medical Re...,Raw data; Scientific and statistical 

In [18]:
# Optional: reorder columns so institution columns are grouped nicely
# (this is just for readability)
cols = [
    "name", "additional_name", "re3_id", "url", "description",
    "contact_email", "year_created", "subjects", "content_types", "keywords",
    "data_license_name", "data_license_url"
]

# Add institution columns in order
max_inst = 0
for col in df_re3_covid.columns:
    if col.startswith("institution_name"):
        idx = int(col.split("institution_name")[1])
        max_inst = max(max_inst, idx)

for i in range(1, max_inst + 1):
    cols.extend([
        f"institution_name{i}",
        f"institution_country{i}",
        f"institution_responsibility_type{i}",
        f"institution_type{i}"
    ])

df_re3_covid = df_re3_covid[cols]

print(df_re3_covid.to_string())

                                                  name                                   additional_name        re3_id                                                                                                                                                                      url                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [19]:
df_re3_covid.to_csv("output/re3_covid_repos.csv", index=False)